In [ ]:
# Cell 1: Imports & Setup
from pathlib import Path
import json, gc
import numpy as np
import polars as pl
import spatial_split as ss

data_base_dir = Path('data/regionwise_yearwise_samples')
output_dir = Path('data/feature_selection_artifacts')
output_dir.mkdir(parents=True, exist_ok=True)

TARGET_REGIONS = ['HIM']          # one region for this run

# Predictor exclusions: metadata, target, and null-heavy composite bands (lat/lon are kept as predictors)
cols_to_drop = [
    'class', 'system:index', 'area_m2', 'area_proportion', 'usable_count',
    '.geo', 'q1_count', 'q2_count', 'q3_count', 'q4_count', 'quarters_present',
    'geometry', 'ndvi_q1_median', 'ndvi_q2_median', 'ndvi_q3_median',
    'ndvi_q4_median', 'tir_count', 'tir_mad', 'tir_median', 'snow_count',
    'year', 'mosaic_version', 'split',
]
pixel_quality_condition = ss.QUALITY          # usable_count >= 5 & quarters_present >= 3
class_remap_expr = ss._remap()                # Level 2 -> Level 1 class remap, same as every other step

MIN_USABLE_LOOKS, MIN_QUARTERS = 5, 3   # (defined in spatial_split.QUALITY)

In [ ]:
# Cell 2: Baseline RF on train-split pixels of the usable years -> feature ranking
from sklearn.ensemble import RandomForestClassifier

for target_region in TARGET_REGIONS:
    print(f"\n{'='*50}\nProcessing region: {target_region}\n{'='*50}")
    years = ss.study_years(target_region)
    assert years, f'run 03_split_and_years.ipynb for {target_region} first'
    print(f"Using {len(years)} usable years: {years[0]}-{years[-1]}")

    lazy_ds = ss.scan_years(target_region, years)
    feature_cols = [c for c in lazy_ds.collect_schema().names() if c not in cols_to_drop]

    df_collected = (
        ss.add_split(lazy_ds.with_columns(class_remap_expr), target_region)
        .filter(pixel_quality_condition & (pl.col('split') == 'train'))
        .select(feature_cols + ['class'])
        .collect()
    )
    print(f"Train-split gated rows: {df_collected.height:,}")
    df_sample = df_collected.sample(n=min(100000, df_collected.height), seed=42)
    del df_collected; gc.collect()

    X_sample = df_sample.select(feature_cols).to_numpy().astype(np.float32)
    y_sample = df_sample['class'].to_numpy().astype(np.int32)
    del df_sample; gc.collect()
    print(f"Sampled baseline array: {X_sample.shape[0]:,} rows, {X_sample.shape[1]} features.")

    rf = RandomForestClassifier(n_estimators=100, max_depth=14, n_jobs=4, random_state=42).fit(X_sample, y_sample)
    imp = dict(zip(feature_cols, [float(v) for v in rf.feature_importances_]))
    all_ranked = sorted(feature_cols, key=lambda c: imp[c], reverse=True)

    out = output_dir / f"{target_region}_ranked_features.json"
    json.dump({"region": target_region, "years": years, "feature_importances": imp, "ranked_features": all_ranked},
              open(out, "w"), indent=2)
    print(f"Saved -> {out}\nTop 10:", all_ranked[:10])
    del X_sample, y_sample, rf; gc.collect()